# 03 · The Hero cascade — d8 global + EBM regime = the 0.12033 ceiling

*Edge-features arc · 01 discovery · 02 linear base · 03 hero cascade · 04 regime-MoE · 05 temporal-kNN  —  machinery: `ridge_pipeline_throughline.ipynb`*

**Verify first.** The deployed stack is a 3-stage cascade: `linbest` base + **d8 global XGB** (Hero A) +
**gated EBM regime** on the h16-19 close/AH leftover (Hero B). Every tree-stage QLIKE below is
**recomputed from the saved per-bar predictions through the real pipeline metric**
(`apply_duan_smearing` → QLIKE) and **asserted** against the cluster collect (`results_all.csv`) — never
pasted. The base row is the linear floor (no local preds), so it carries its exact **reproduce-command**
instead. Machinery = `resid_amortized.preds_chunk` (`resid_regime` arm, the through-line for ch. 04).

In [1]:
import html, inspect, json, os, sys, textwrap
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

def find_repo(s):
    for q in [Path(s).resolve(), *Path(s).resolve().parents]:
        if (q / "resid_amortized.py").exists() and (q / "src").is_dir():
            return q
    raise FileNotFoundError("repo root")
REPO = find_repo(Path.cwd()); os.chdir(REPO); sys.path.insert(0, str(REPO))

from src.evaluation.metrics import apply_duan_smearing  # the REAL pipeline metric

# Collapsible, theme-following source display (one <details> per object; folded).
def _details(f, open_=False):
    mod = f.__module__.replace("src.", "src/").replace(".", "/") + ".py"
    try:
        sig = ("class " + f.__name__) if inspect.isclass(f) else ("def " + f.__name__ + str(inspect.signature(f)))
    except (ValueError, TypeError):
        sig = f.__qualname__
    body = "```python\n" + textwrap.dedent(inspect.getsource(f)).rstrip() + "\n```"
    return (f"<details{' open' if open_ else ''}>\n<summary><code>{html.escape(mod + '  ·  ' + sig)}"
            f"</code></summary>\n\n{body}\n\n</details>")
def show_one(f):
    return Markdown(_details(f))

# ── the verification machinery: recompute QLIKE from saved per-bar preds ──────
# Local preds are gitignored; if absent we print the exact reproduce-command.
PREDS = REPO / "results" / "moe_ladder" / "preds"
CID = "xgb_all_buckets_tw1000_enetreg2_linbest_rf480_slim"

def recompute(label):
    """Full-OOS QLIKE recomputed from saved per-bar preds via apply_duan_smearing.
    Returns (qlike, n) if the preds csv is present, else (None, 0)."""
    fp = PREDS / f"{label}.csv"
    if not fp.exists():
        return None, 0
    df = pd.read_csv(fp)                                  # cols: k, pred_adj, y_true, base
    pr, tr = apply_duan_smearing(df.pred_adj.to_numpy(), df.y_true.to_numpy(), df.base.to_numpy())
    m = (tr > 0) & (pr > 0); r = tr[m] / pr[m]
    return float(np.mean(r - np.log(r) - 1.0)), int(len(df))

def reproduce_cmd(arm, label):
    return f"$PY resid_amortized.py chunk_collect {CID} {arm} {label}"

# cluster reference (results_all.csv) — what each RECOMPUTE must reproduce to <1e-4
REF = pd.read_csv(REPO / "results" / "moe_ladder" / "results_all.csv").set_index("label")
print("setup ok | preds present:", PREDS.exists(), "| reference rows:", len(REF))

setup ok | preds present: True | reference rows: 11


---
## 1 · Verify — the cascade ladder, recomputed

The thin through-line: each tree-stage row reloads its saved per-bar predictions and runs the **exact**
pipeline QLIKE on them — `apply_duan_smearing(pred_adj, y_true, base)` → masked `ratio − log ratio − 1`
mean (folded below). The recompute is asserted against the cluster `results_all.csv` to `<1e-4`. The base
(`enetreg2_linbest`, the linear floor) is not a residual collect and has no local preds, so its row shows
the **reproduce-command** and reads its value from the recorded `linbest_ladder.csv` artifact (not pasted
into this cell).

In [2]:
show_one(apply_duan_smearing)   # the real metric — folded, recomputed below

<details>
<summary><code>src/evaluation/metrics.py  ·  def apply_duan_smearing(forecasts: &#x27;np.ndarray&#x27;, y_true: &#x27;np.ndarray&#x27;, baselines: &#x27;np.ndarray&#x27;) -&gt; &#x27;tuple[np.ndarray, np.ndarray]&#x27;</code></summary>

```python
def apply_duan_smearing(
    forecasts: np.ndarray,
    y_true: np.ndarray,
    baselines: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Apply Duan smearing correction to convert adjusted-scale forecasts to raw scale.

    Parameters
    ----------
    forecasts : array-like
        Model predictions on adjusted (sqrt / log) scale.
    y_true : array-like
        True values on adjusted scale.
    baselines : array-like
        Baseline volatility used to scale back to raw units.

    Returns
    -------
    pred_raw : np.ndarray
        Smearing-corrected predictions on raw scale.
    true_raw : np.ndarray
        True values on raw scale.
    """
    forecasts = np.asarray(forecasts, dtype=np.float64)
    y_true = np.asarray(y_true, dtype=np.float64)
    baselines = np.asarray(baselines, dtype=np.float64)

    smear = np.mean((y_true - forecasts) ** 2)
    pred_raw = (forecasts**2 + smear) * baselines
    true_raw = (y_true**2) * baselines
    return pred_raw, true_raw
```

</details>

In [3]:
# base value comes from the recorded ladder artifact, not hardcoded here
LL = pd.read_csv(REPO / "results" / "moe_ladder" / "linbest_ladder.csv")
base_q = float(LL.loc[LL.label == "enetreg2_linbest", "qlike"].iloc[0])

# (stage, preds-label | None, arm, note).  None preds-label ⇒ cluster-only ⇒ reproduce-cmd.
LADDER = [
    ("base  enetreg2_linbest",          None,       "residualized", "linear floor — no local preds"),
    ("+ d8@cs0.5 global XGB (Hero A)",   "fa_d8c5",  "heroA",        "global tree, no regime stage"),
    ("+ tuned XGB regime (Hero B′)",     "fb_r3n4",  "resid_regime", "best curated XGB regime (depth 3)"),
    ("+ EBM regime  =  THE CEILING",     "ctrl_ebm", "resid_regime", "EBM regime — the bar (== lbHeroB)"),
]
rows = []
for stage, lbl, arm, note in LADDER:
    if lbl is None:
        rows.append({"stage": stage, "qlike": round(base_q, 5), "n": -1,
                     "source": "cluster (reproduce-cmd)", "ref/cmd": reproduce_cmd(arm, "<linbest base>"),
                     "note": note})
        continue
    q, n = recompute(lbl)
    assert q is not None, f"{lbl}: preds missing — reproduce with  {reproduce_cmd(arm, lbl)}"
    ref = float(REF.loc[lbl, "qlike_full"])
    assert abs(q - ref) < 1e-4, f"{lbl}: RECOMPUTE {q:.5f} != cluster {ref:.5f}"
    rows.append({"stage": stage, "qlike": round(q, 5), "n": n,
                 "source": "RECOMPUTE", "ref/cmd": f"== results_all {ref:.5f}", "note": note})

# anchor identity: ctrl_ebm must reproduce the INDEPENDENT lbHeroB collect
q_ctrl, _ = recompute("ctrl_ebm"); q_lb, _ = recompute("lbHeroB")
assert q_ctrl is not None and q_lb is not None, "anchor preds missing"
assert abs(q_ctrl - q_lb) < 1e-4, f"ctrl_ebm {q_ctrl:.5f} != lbHeroB {q_lb:.5f}"

ladder = pd.DataFrame(rows)
display(ladder[["stage", "qlike", "source", "ref/cmd", "n", "note"]])
print(f"PASS — every RECOMPUTE row reproduces its cluster QLIKE to <1e-4; "
      f"ctrl_ebm reproduces the independent lbHeroB collect ({q_ctrl:.5f} vs {q_lb:.5f}).")
print(f"Ceiling = {q_ctrl:.5f}  (linbest + d8@cs0.5 global + EBM regime). ch.04 targets exactly this slot.")
print(f"Base (linear floor, no preds) — reproduce: {reproduce_cmd('residualized', '<linbest base>')}")

,stage,qlike,source,ref/cmd,n,note
0,base enetreg2_linbest,0.12266,cluster (reproduce-cmd),$PY resid_amortized.py chunk_collect xgb_all_b...,-1,linear floor — no local preds
1,+ d8@cs0.5 global XGB (Hero A),0.12081,RECOMPUTE,== results_all 0.12081,194934,"global tree, no regime stage"
2,+ tuned XGB regime (Hero B′),0.12072,RECOMPUTE,== results_all 0.12072,194934,best curated XGB regime (depth 3)
3,+ EBM regime = THE CEILING,0.12033,RECOMPUTE,== results_all 0.12033,194934,EBM regime — the bar (== lbHeroB)


PASS — every RECOMPUTE row reproduces its cluster QLIKE to <1e-4; ctrl_ebm reproduces the independent lbHeroB collect (0.12033 vs 0.12033).
Ceiling = 0.12033  (linbest + d8@cs0.5 global + EBM regime). ch.04 targets exactly this slot.
Base (linear floor, no preds) — reproduce: $PY resid_amortized.py chunk_collect xgb_all_buckets_tw1000_enetreg2_linbest_rf480_slim residualized <linbest base>


---
## 2 · Interpret — the EBM regime is the *correct* learner, not a compromise

The ladder is an **expressivity stack that subsumes upward**: each stage only earns its keep where the
stage below it left structure on the table.

- **The global d8 tree subsumes the linear base.** linbest `0.12266` → +d8@cs0.5 global **`0.12081`**
  (−0.00185). One global XGB absorbs the bulk of the residual nonlinearity the penalized-linear base
  could not — the famous edge is a *global* tree effect, not a regime one.
- **The XGB regime barely moves, and LOSES to the EBM.** A tuned XGB *regime* stage on the h16-19
  close/AH leftover gives only **`0.12072`** (−0.00009 vs no regime), and the proper **120-trial Optuna**
  search lands at 0.12094 — *worse than doing nothing* (the no-regime global, 0.12081). On the
  few-thousand-row, ~93%-noise h16-19 subset a powerful booster overfits; confirmed by optimization,
  not assertion.
- **The EBM regime is the right regularizer.** Swapping XGB→EBM on the *same* slot gives **`0.12033`**
  (−0.00048 vs no-regime), reproducing the independent `lbHeroB` collect. The EBM's
  **additive + pairwise + bagging** is genuinely the stronger small-sample learner here.
- **The floor is real.** Every post-floor lever died for the *right* reason: the **log-signature**
  antisymmetric path basis is null (the close edge has **no chronological-order content** — a pure
  state/level regime), the turnover-**OFI** proxy is null, and five **regime-persistence** families are
  null OOS (best −0.00002; the cleanest in-sample axis, the fast/slow vol cascade, is 87% spanned).

⇒ **`0.12033` is the ceiling** of the price-only cascade. The two open levers are **(1) new data**
(auction imbalance / GEX) and **(2) DL on the linbest residual** targeting state-space (not path)
nonlinearity — exactly what ch. 04 tests by swapping the regime learner for a learned mixture-of-experts.